# LLM Usage Tutorial

This notebook demonstrates how to use the `LLMClient` provided by the `kavalai` package. `LLMClient` provides a unified interface for interacting with different LLM providers (currently OpenAI and Gemini) while handling metrics collection, cost calculation, and structured output.


## Setup

First, let's import the necessary classes and ensure we have an API key set up in our environment (e.g., `OPENAI_API_KEY` or `GEMINI_API_KEY`).


In [2]:
import asyncio
from pydantic import BaseModel
from kavalai import LLMClient, Streamer, StreamContent
import dotenv

# Load API keys from .env file.
dotenv.load_dotenv()

# Or set your API key if not already in environment
# os.environ["OPENAI_API_KEY"] = "your-key-here"
# os.environ["GEMINI_API_KEY"] = "your-key-here"


True

## Basic Chat Completion

The simplest way to use `LLMClient` is for basic text completions. `chat_completions` returns a tuple containing the result and execution stats.


In [4]:
async def basic_example():
    # Initialize client with a specific model (provider/model-name)
    client = LLMClient(model="openai/gpt-4o-mini")
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are three interesting facts about Estonia?"}
    ]

    result, stats = await client.chat_completions(messages=messages)
    
    print(f"Result:\n{result}\n")
    print(f"Stats: {stats.total_tokens} tokens, duration: {stats.duration_seconds:.2f}s")

await basic_example()


Result:
Sure! Here are three interesting facts about Estonia:

1. **Digital Innovation**: Estonia is known for being one of the most digitally advanced countries in the world. It was the first country to offer e-residency, allowing global citizens to start and manage businesses online. The country also has a fully digital ID system that enables citizens to access a wide range of government services online.

2. **Nature and Clean Air**: Approximately 50% of Estonia is covered by forest, making it one of the greenest countries in Europe. It is home to many national parks, nature reserves, and diverse wildlife. Estonia also boasts some of the cleanest air in the world, a testament to its commitment to environmental conservation.

3. **Rich Cultural Heritage**: Estonia has a vibrant cultural scene, influenced by its history and various ethnic groups, including the Finno-Ugric roots from Finland. The country celebrates numerous traditional festivals, such as the Song and Dance Festival, whi

## Structured Output (Pydantic)

Kaval.AI supports structured output by passing a Pydantic model to `response_model`. The LLM will guarantee that the output matches this schema.


In [5]:
class Fact(BaseModel):
    topic: str
    fact: str
    relevance_score: float

class FactsList(BaseModel):
    facts: list[Fact]

async def structured_example():
    client = LLMClient(model="gemini/gemini-2.0-flash")
    messages = [
        {"role": "user", "content": "Provide 3 interesting facts about chess."}
    ]

    # Passing response_model returns an instance of that model
    result, stats = await client.chat_completions(
        messages=messages, 
        response_model=FactsList
    )
    
    for fact in result.facts:
        print(f"[{fact.topic}] ({fact.relevance_score}): {fact.fact}")

await structured_example()


[Chess History] (0.8): The longest chess game theoretically possible is 5,949 moves.
[Chess Strategies] (0.7): The most common opening move in chess is advancing the king's pawn two squares (e4).
[Chess Records] (0.9): Garry Kasparov was the world's highest-rated chess player for a record 255 months overall.


## Streaming

For real-time responses, you can use a `Streamer` and an `asyncio.Queue`.


In [5]:
PROMPT = """
Count from 1 to 10, say

a. "FizzBuzz" if i is divisible by 3 and 5.
b. "Fizz" if i is divisible by 3.
c. "Buzz" if i is divisible by 5.
"""

async def streaming_example():
    client = LLMClient(model="openai/gpt-4o-mini")
    messages = [{"role": "user", "content": PROMPT}]
    queue = asyncio.Queue()
    streamer = Streamer("response", queue)

    # Run the client call as a task to consume chunks from the queue
    task = asyncio.create_task(
        client.chat_completions(messages=messages, streamer=streamer, stream_delta=True)
    )

    while True:
        raw_chunk = await queue.get()
        chunk = StreamContent.model_validate_json(raw_chunk)
        
        if chunk.type == "partial":
            print(chunk.value, end="", flush=True)
        elif chunk.type == "complete":
            break
    
    _, stats = await task
    print(f"\n\nTokens: {stats.total_tokens}")

await streaming_example()


Here's the count from 1 to 10 following your rules:

1. 1
2. 2
3. Fizz
4. 4
5. Buzz
6. Fizz
7. 7
8. 8
9. Fizz
10. Buzz

Let me know if you need anything else!

Tokens: 132


## Embeddings

You can also compute text embeddings for tasks like RAG (Retrieval Augmented Generation).
By default we are using `openai/text-embedding-3-small` model, that is designed for text embedding tasks and is optimized for performance and accuracy.
It is a small model that is suitable for use even in production environments.


In [7]:
async def embeddings_example():
    client = LLMClient(model="openai/text-embedding-3-small")
    texts = ["Kaval.AI is an open source AI agent toolkit.",
             "D minor and F major are parallel keys."]
    
    embeddings, stats = await client.compute_embeddings(texts=texts, normalize=True)
    
    print(f"Number of embeddings: {len(embeddings)}")
    print(f"Embedding dimension: {len(embeddings[0])}\n")
    print(f"Tokens: {stats.total_tokens}")

await embeddings_example()


Number of embeddings: 2
Embedding dimension: 1536

Tokens: 20
